In [1]:
# Load
import pandas as pd
import numpy as np

# Load critical customers identified in the previous analysis
critical_customers = pd.read_csv(
    "Data/critical_customers.csv"
)

critical_customers.head()

,customer_id,last_order_date,total_orders,total_revenue,recency_days,recency_segment,revenue_percentile,recency_risk_score,value_score,frequency_score,priority_score,priority_segment
0,CUST019255,2025-01-02,9,118694.06,362,Inactive,0.993343,83.257527,99.334328,87.850899,88.999242,Critical Priority
1,CUST000527,2025-02-01,11,100754.20,332,Inactive,0.986000,80.707365,98.599957,93.017554,88.537180,Critical Priority
2,CUST021374,2024-09-20,9,48963.63,466,Inactive,0.864781,89.982944,86.478099,87.850899,88.505081,Critical Priority
3,CUST023638,2025-02-10,10,102612.83,323,Inactive,0.987160,79.869945,98.716035,90.745742,87.698932,Critical Priority
4,CUST022178,2025-02-26,12,96931.92,307,Inactive,0.983394,78.235378,98.339374,94.771753,87.573852,Critical Priority


In [2]:
# Create Customer Action Priority
# Create actionable priority levels

def get_action_priority(score):
    if score >= 85:
        return "Immediate Action"
    elif score >= 80:
        return "High Priority"
    elif score >= 70:
        return "Medium Priority"
    else:
        return "Low Priority"


critical_customers["action_priority"] = (
    critical_customers["priority_score"]
    .apply(get_action_priority)
)

critical_customers[
    [
        "customer_id",
        "total_revenue",
        "recency_days",
        "priority_score",
        "priority_segment",
        "action_priority"
    ]
].head(10)

,customer_id,total_revenue,recency_days,priority_score,priority_segment,action_priority
0,CUST019255,118694.06,362,88.999242,Critical Priority,Immediate Action
1,CUST000527,100754.20,332,88.537180,Critical Priority,Immediate Action
2,CUST021374,48963.63,466,88.505081,Critical Priority,Immediate Action
3,CUST023638,102612.83,323,87.698932,Critical Priority,Immediate Action
4,CUST022178,96931.92,307,87.573852,Critical Priority,Immediate Action
5,CUST049785,47523.36,419,87.414541,Critical Priority,Immediate Action
6,CUST047664,62362.05,393,87.306635,Critical Priority,Immediate Action
7,CUST043927,55820.01,414,87.193045,Critical Priority,Immediate Action
8,CUST030558,77339.89,344,86.550115,Critical Priority,Immediate Action
9,CUST047284,61035.40,441,86.497761,Critical Priority,Immediate Action


In [3]:
# Assign a recommended business action

def recommend_action(row):

    if row["recency_days"] >= 365 and row["total_revenue"] >= 50000:
        return "High-Value Win-Back Campaign"

    elif row["recency_days"] >= 300 and row["total_revenue"] >= 50000:
        return "Urgent Retention Campaign"

    elif row["recency_days"] >= 300:
        return "Re-Engagement Campaign"

    elif row["total_revenue"] >= 50000:
        return "VIP Retention Campaign"

    else:
        return "Personalized Promotion"


critical_customers["recommended_action"] = (
    critical_customers.apply(recommend_action, axis=1)
)

critical_customers[
    [
        "customer_id",
        "total_revenue",
        "recency_days",
        "priority_score",
        "action_priority",
        "recommended_action"
    ]
].head(15)

,customer_id,total_revenue,recency_days,priority_score,action_priority,recommended_action
0,CUST019255,118694.06,362,88.999242,Immediate Action,Urgent Retention Campaign
1,CUST000527,100754.20,332,88.537180,Immediate Action,Urgent Retention Campaign
2,CUST021374,48963.63,466,88.505081,Immediate Action,Re-Engagement Campaign
3,CUST023638,102612.83,323,87.698932,Immediate Action,Urgent Retention Campaign
4,CUST022178,96931.92,307,87.573852,Immediate Action,Urgent Retention Campaign
5,CUST049785,47523.36,419,87.414541,Immediate Action,Re-Engagement Campaign
6,CUST047664,62362.05,393,87.306635,Immediate Action,High-Value Win-Back Campaign
7,CUST043927,55820.01,414,87.193045,Immediate Action,High-Value Win-Back Campaign
8,CUST030558,77339.89,344,86.550115,Immediate Action,Urgent Retention Campaign
9,CUST047284,61035.40,441,86.497761,Immediate Action,High-Value Win-Back Campaign


In [4]:
critical_customers.columns.tolist()

['customer_id',
 'last_order_date',
 'total_orders',
 'total_revenue',
 'recency_days',
 'recency_segment',
 'revenue_percentile',
 'recency_risk_score',
 'value_score',
 'frequency_score',
 'priority_score',
 'priority_segment',
 'action_priority',
 'recommended_action']

In [5]:
# Load the customer-level category recommendations
retention_recommendations = pd.read_csv(
    "Data/customer_retention_actions.csv"
)

retention_recommendations.head()

,customer_id,total_orders,total_revenue,recency_days,priority_score,acquisition_channel,preferred_category,category_revenue,customer_revenue_share,preference_strength,recommended_strategy
0,CUST019255,9,118694.06,362,88.999242,Organic Search,Beauty,44190.10,35.673858,Mixed,Immediate - Broad personalized re-engagement
1,CUST000527,11,100754.20,332,88.537180,Instagram,Beauty,44778.47,43.709002,Mixed,Immediate - Broad personalized re-engagement
2,CUST021374,9,48963.63,466,88.505081,Organic Search,Electronics,23174.24,44.505815,Mixed,Immediate - Broad personalized re-engagement
3,CUST023638,10,102612.83,323,87.698932,Referral,Fashion,48330.86,43.503158,Mixed,High - Broad personalized re-engagement
4,CUST022178,12,96931.92,307,87.573852,Organic Search,Sports,28109.92,28.297248,Mixed,High - Broad personalized re-engagement


In [6]:
retention_recommendations.columns.tolist()

['customer_id',
 'total_orders',
 'total_revenue',
 'recency_days',
 'priority_score',
 'acquisition_channel',
 'preferred_category',
 'category_revenue',
 'customer_revenue_share',
 'preference_strength',
 'recommended_strategy']

In [7]:
# Merge the action recommendations
# Select only the action information we created in this notebook

action_data = critical_customers[
    [
        "customer_id",
        "action_priority",
        "recommended_action"
    ]
].copy()

action_data.head()

,customer_id,action_priority,recommended_action
0,CUST019255,Immediate Action,Urgent Retention Campaign
1,CUST000527,Immediate Action,Urgent Retention Campaign
2,CUST021374,Immediate Action,Re-Engagement Campaign
3,CUST023638,Immediate Action,Urgent Retention Campaign
4,CUST022178,Immediate Action,Urgent Retention Campaign


In [8]:
# Combine customer intelligence with recommended actions

final_recommendations = retention_recommendations.merge(
    action_data,
    on="customer_id",
    how="left"
)

final_recommendations.head()

,customer_id,total_orders,total_revenue,recency_days,priority_score,acquisition_channel,preferred_category,category_revenue,customer_revenue_share,preference_strength,recommended_strategy,action_priority,recommended_action
0,CUST019255,9,118694.06,362,88.999242,Organic Search,Beauty,44190.10,35.673858,Mixed,Immediate - Broad personalized re-engagement,Immediate Action,Urgent Retention Campaign
1,CUST000527,11,100754.20,332,88.537180,Instagram,Beauty,44778.47,43.709002,Mixed,Immediate - Broad personalized re-engagement,Immediate Action,Urgent Retention Campaign
2,CUST021374,9,48963.63,466,88.505081,Organic Search,Electronics,23174.24,44.505815,Mixed,Immediate - Broad personalized re-engagement,Immediate Action,Re-Engagement Campaign
3,CUST023638,10,102612.83,323,87.698932,Referral,Fashion,48330.86,43.503158,Mixed,High - Broad personalized re-engagement,Immediate Action,Urgent Retention Campaign
4,CUST022178,12,96931.92,307,87.573852,Organic Search,Sports,28109.92,28.297248,Mixed,High - Broad personalized re-engagement,Immediate Action,Urgent Retention Campaign


In [9]:
final_recommendations.columns.tolist()

['customer_id',
 'total_orders',
 'total_revenue',
 'recency_days',
 'priority_score',
 'acquisition_channel',
 'preferred_category',
 'category_revenue',
 'customer_revenue_share',
 'preference_strength',
 'recommended_strategy',
 'action_priority',
 'recommended_action']

In [10]:
# Generate a Personalized Business Recommendation
# Now we will turn those into a human-readable recommendation.
def generate_recommendation(row):

    category = row["preferred_category"]
    action = row["recommended_action"]
    strength = row["preference_strength"]

    if strength == "Strong":
        return (
            f"{action} focused on {category}. "
            f"Customer shows a strong preference for this category."
        )

    elif strength == "Moderate":
        return (
            f"{action} with a {category}-focused offer. "
            f"Use personalized recommendations to encourage re-purchase."
        )

    else:
        return (
            f"{action} with personalized {category} recommendations. "
            f"Test multiple offers to identify the strongest response."
        )


final_recommendations["business_recommendation"] = (
    final_recommendations.apply(
        generate_recommendation,
        axis=1
    )
)

In [11]:
final_recommendations[
    [
        "customer_id",
        "total_revenue",
        "recency_days",
        "priority_score",
        "preferred_category",
        "preference_strength",
        "recommended_action",
        "business_recommendation"
    ]
].head(15)

,customer_id,total_revenue,recency_days,priority_score,preferred_category,preference_strength,recommended_action,business_recommendation
0,CUST019255,118694.06,362,88.999242,Beauty,Mixed,Urgent Retention Campaign,Urgent Retention Campaign with personalized Be...
1,CUST000527,100754.20,332,88.537180,Beauty,Mixed,Urgent Retention Campaign,Urgent Retention Campaign with personalized Be...
2,CUST021374,48963.63,466,88.505081,Electronics,Mixed,Re-Engagement Campaign,Re-Engagement Campaign with personalized Elect...
3,CUST023638,102612.83,323,87.698932,Fashion,Mixed,Urgent Retention Campaign,Urgent Retention Campaign with personalized Fa...
4,CUST022178,96931.92,307,87.573852,Sports,Mixed,Urgent Retention Campaign,Urgent Retention Campaign with personalized Sp...
5,CUST049785,47523.36,419,87.414541,Home & Kitchen,Mixed,Re-Engagement Campaign,Re-Engagement Campaign with personalized Home ...
6,CUST047664,62362.05,393,87.306635,Fashion,Moderate,High-Value Win-Back Campaign,High-Value Win-Back Campaign with a Fashion-fo...
7,CUST043927,55820.01,414,87.193045,Beauty,Mixed,High-Value Win-Back Campaign,High-Value Win-Back Campaign with personalized...
8,CUST030558,77339.89,344,86.550115,Electronics,Mixed,Urgent Retention Campaign,Urgent Retention Campaign with personalized El...
9,CUST047284,61035.40,441,86.497761,Electronics,Moderate,High-Value Win-Back Campaign,High-Value Win-Back Campaign with a Electronic...


In [12]:
# Rank the customers for the business team
action_queue = (
    final_recommendations
    .sort_values(
        ["priority_score", "total_revenue"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

action_queue.index = action_queue.index + 1
action_queue.index.name = "action_rank"

action_queue.head(15)

,customer_id,total_orders,total_revenue,recency_days,priority_score,acquisition_channel,preferred_category,category_revenue,customer_revenue_share,preference_strength,recommended_strategy,action_priority,recommended_action,business_recommendation
action_rank,,,,,,,,,,,,,,
1,CUST019255,9,118694.06,362,88.999242,Organic Search,Beauty,44190.10,35.673858,Mixed,Immediate - Broad personalized re-engagement,Immediate Action,Urgent Retention Campaign,Urgent Retention Campaign with personalized Be...
2,CUST000527,11,100754.20,332,88.537180,Instagram,Beauty,44778.47,43.709002,Mixed,Immediate - Broad personalized re-engagement,Immediate Action,Urgent Retention Campaign,Urgent Retention Campaign with personalized Be...
3,CUST021374,9,48963.63,466,88.505081,Organic Search,Electronics,23174.24,44.505815,Mixed,Immediate - Broad personalized re-engagement,Immediate Action,Re-Engagement Campaign,Re-Engagement Campaign with personalized Elect...
4,CUST023638,10,102612.83,323,87.698932,Referral,Fashion,48330.86,43.503158,Mixed,High - Broad personalized re-engagement,Immediate Action,Urgent Retention Campaign,Urgent Retention Campaign with personalized Fa...
5,CUST022178,12,96931.92,307,87.573852,Organic Search,Sports,28109.92,28.297248,Mixed,High - Broad personalized re-engagement,Immediate Action,Urgent Retention Campaign,Urgent Retention Campaign with personalized Sp...
6,CUST049785,10,47523.36,419,87.414541,Affiliate,Home & Kitchen,18042.28,29.260753,Mixed,High - Broad personalized re-engagement,Immediate Action,Re-Engagement Campaign,Re-Engagement Campaign with personalized Home ...
7,CUST047664,8,62362.05,393,87.306635,Email,Fashion,46692.46,64.348408,Moderate,High - Category-focused re-engagement,Immediate Action,High-Value Win-Back Campaign,High-Value Win-Back Campaign with a Fashion-fo...
8,CUST043927,8,55820.01,414,87.193045,Organic Search,Beauty,20830.55,37.317353,Mixed,High - Broad personalized re-engagement,Immediate Action,High-Value Win-Back Campaign,High-Value Win-Back Campaign with personalized...
9,CUST030558,8,77339.89,344,86.550115,Referral,Electronics,27178.52,35.141658,Mixed,High - Broad personalized re-engagement,Immediate Action,Urgent Retention Campaign,Urgent Retention Campaign with personalized El...


CUST019255 is ranked #1, with ₹118,694 historical revenue, 362 days of inactivity, and a priority score of 88.999.


In [16]:
# Calculate Revenue at Risk

# historical revenue associated with the Critical customers.
revenue_at_risk = action_queue["total_revenue"].sum()

print(
    f"Historical revenue associated with Critical customers: "
    f"₹{revenue_at_risk:,.2f}"
)

Historical revenue associated with Critical customers: ₹1,867,067.66


In [17]:
# Then calculate the number of customers:
critical_customer_count = action_queue["customer_id"].nunique()

print(
    f"Critical customers: {critical_customer_count:,}"
)

Critical customers: 26


In [ ]:

average_revenue_per_critical_customer = (
    revenue_at_risk / critical_customer_count
)

print(
    f"Average historical revenue per Critical customer: "
    f"₹{average_revenue_per_critical_customer:,.2f}"
)

Average historical revenue per Critical customer: ₹71,810.29
